### 1. OpenAI Chat Completions API 구조
- **메시지 역할(Role)**: `system` (기본 지시사항), `user` (사용자 입력), `assistant` (LLM의 응답)
- **주요 파라미터**: `temperature` (응답의 무작위성 제어, ReAct에서는 0에 가깝게 설정하는 것이 유리), `max_tokens` 등
- **API 호출 구조**: `openai` 파이썬 패키지를 이용한 기본 호출 방법 소개

In [7]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client  = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
# test
response = client.chat.completions.create(
    model='gpt-5.4-nano',
    messages=[{'role':'user','content':'오늘의 날씨정보를 검색해서 알려주세요'}],
    temperature=0,    
)
print(response.choices[0].message.content)
print(f'토큰사용량 : {response.usage}')

죄송하지만, 저는 현재 **실시간 웹 검색으로 오늘의 날씨를 직접 조회**할 수는 없어요. 대신 아래 정보를 주시면, 그 지역의 **날씨를 확인하는 방법**이나 **날씨 해석(예: 우산 필요 여부)**을 바로 도와드릴게요.

1) **지역(도시/구/동)**: 예) 서울 강남구, 부산 해운대구  
2) **원하는 시간대**: 지금 / 오늘(오전·오후) / 시간대별  
3) **원하는 항목**: 기온, 강수(비/눈), 미세먼지, 바람, 체감온도 등

원하시면, 사용 중인 기기 기준으로 바로 확인할 수 있는 링크/방법도 안내해드릴까요? (예: 네이버/기상청/카카오맵/구글 날씨)
토큰사용량 : CompletionUsage(completion_tokens=198, prompt_tokens=16, total_tokens=214, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


In [6]:
response.usage.total_tokens

83

### 2. ReAct 시스템 프롬프트 설계
시스템 프롬프트에는 다음이 반드시 포함되어야 합니다:
- 에이전트의 역할 지정
- 사용 가능한 도구(Tool) 목록과 그 사용법
- 반드시 따라야 하는 출력 형식 (Thought -> Action -> Observation 패턴)

### 3. ReAct 루프 직접 구현
- **while 또는 for 루프 기반 에이전트 구조**: 에이전트가 `Finish` 액션을 낼 때까지 계속해서 응답을 생성하고 도구를 실행하는 과정을 반복합니다.
- **종료 조건**: 모델의 출력에서 `Action: Finish`가 감지되면 무한 루프를 종료하고 최종 답을 반환합니다.
- **최대 반복 횟수 제한**: 무한 루프를 방지하기 위해 최대 스텝 수를 지정합니다.

In [ ]:
import re
class SimpleReactAgent:    
    def __init__(self,model = 'gpt-5.4-nano', max_steps=5):        
        self.model = model
        self.max_steps = max_steps
        self.tools = {}
        self.systemp_prompt = ''
    def register_tool(self, name,func,description):
        self.tools[name] = {'func':func, 'description':description}
    def _build_system_prompt(self):
        tool_desc = '\n'.join([ f"- {name}:{info['description']}" for name, info in self.tools.items()  ])
        self.systemp_prompt = f'''당신은 주어진 질문에 정확하게 답변하기 위해 도구를 사용하는 AI 에이전트입니다.
        사용 가능한 도구:
        {tool_desc}

        반드시 다음 형식을 따라 응답하세요:
        Thought: [현재 상황에 대한 추론]
        Action: [도구이름][입력값]

        도구 실행결과는 Observation으로 제공됩니다
        최종 답변을 제출할 때는 반드시 다음 형식을 사용하세요:
        Thought: [최종 추론]
        Action: Finish[최종 답변]

        중요: 한 번에 하나의 Action만 수행하세요
        '''
    def _parse_action(self, text):
        match = re.search(r'Action\s*:\s*(\w+)\[(.+?)\]', text, re.DOTALL)
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return None, None
    




